# 03 - TimesFM 2.5 zero-shot comparison

**Goal:** test whether a pretrained time-series foundation model, given only your past values and *no training*,
forecasts tomorrow's sleep as well as your LightGBM.

**Concepts**
- *Zero-shot*: the model was pretrained on a huge corpus of series and is applied unchanged. No fitting, so no
  overfitting to your 110 rows, which is exactly why it is interesting here.
- *Context window*: you hand it the last `N` values; it returns point and quantile forecasts for the next `H`.
- *Quantiles*: a forecast band (for example 10th and 90th percentile). Check *coverage*: does the actual land inside
  the band about 80% of the time?
- *Univariate*: it sees one series. It cannot use alcohol, HRV or weekday unless you use covariate features
  (a later experiment), so do not expect it to explain anything. It only forecasts.

**Practical limits (read before installing)**
- ~200M parameters: roughly 1 GB of weights and a PyTorch install. Run this on your laptop, not the VPS
  (5 GB disk free, shared 4 GB RAM). Production use only happens if it wins *and* fits.
- The API changed between TimesFM releases. Follow the README of the version you install; the sketch below may
  be out of date.

**Rule (CLAUDE.md #3):** TimesFM cannot take NaN. If you fill gaps, do it in a *named* function, record which days
were filled, and report results with and without them. Never fill silently.

In [ ]:
# --- Setup (provided, working). Do not edit; fix things in api/app/ml/features.py instead. ---
import pathlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "api").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "api"))

from app.ml import features, evaluate          # the SAME modules the nightly job imports

raw = features.frame_from_csv(ROOT / "ml/data/daily_metrics.csv")   # one row per calendar day
feat = features.build(raw)                                          # baselines, z-scores, missing flags
print(f"{len(raw)} days, {raw.index.min().date()} -> {raw.index.max().date()}, "
      f"{int(raw['row_present'].sum())} with a row, {int(raw['sleep_hours'].isna().sum())} with no sleep value")
raw.tail()

## 1. Install and load
```
pip install timesfm   # check the 2.5 README for the exact package, torch version and checkpoint name
```
Rough shape of the 2.5 API (verify!): load a pretrained checkpoint, `compile` a `ForecastConfig` with your max
context and horizon, then `forecast(horizon=..., inputs=[np.array(...)])` returns `(point, quantiles)`.

In [ ]:
# TODO: import timesfm; load the model once (this downloads weights). Keep it in a variable.

## 2. Prepare the context series
Decide and write down: how many past days as context, and what you do about NaN days.
Option to evaluate: forecast only on dates where the last `k` days are complete (skips the problem, but changes
which days you score; compare baselines on the *same* subset).

In [ ]:
# TODO: a named function that returns the context array for date t using only data <= t, and which days it filled (if any).

## 3. Walk-forward zero-shot forecasts
For each of the last 30 scorable dates `t`: give it context up to `t`, ask for 1 day ahead, store the point forecast and the 10th/90th quantiles.

In [ ]:
# TODO: loop over dates; keep results in a DataFrame indexed by t. Batch the calls: the model accepts a list of series.

## 4. Compare on identical dates
Use the same 30 target dates for `naive`, `mean7`, `LightGBM` and `TimesFM`. `evaluate.mae` and `evaluate.skill`.

In [ ]:
# TODO: one table; then a plot of actual vs each model's forecast.

## 5. Calibration of the band
What fraction of actuals fall between the 10th and 90th percentile? About 0.8 is well calibrated. Much less means overconfident intervals.

In [ ]:
# TODO: coverage = mean((actual >= q10) & (actual <= q90))

## Decision (fill in)
| Model | MAE (h) | n | Skill vs naive | Interval coverage | Fits VPS? |
|---|---|---|---|---|---|
| naive | | | 0 | n/a | yes |
| mean7 | | | | n/a | yes |
| LightGBM | | | | | yes |
| TimesFM 2.5 | | | | | |

**Which one should the nightly job run, and why?** (Remember: promote only if it beats the incumbent on the last 30 days.)